In [ ]:
from scipy.io import loadmat
from sklearn.decomposition import PCA
import numpy as np
from scipy.stats import multivariate_normal as mvn
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.cluster import KMeans

The demo code file Module7_EM-gmm.py was utitlized heavily for this question 

In [ ]:
def calc_pdf(data, mu, sigma):
    d = len(mu)
    data_centered = data - mu

    inv_sigma = np.linalg.inv(sigma)
    sign, logdet = np.linalg.slogdet(sigma) #https://www.geeksforgeeks.org/python/compute-the-sign-and-natural-logarithm-of-the-determinant-of-an-array-in-python/

    # exponent = np.sum((data_centered @ inv_sigma) * data_centered, axis=1)
    exponent = np.sum((data_centered @ inv_sigma) * data_centered)
    log_pdf = -0.5 * (d*np.log(2*np.pi) + logdet + exponent)

    return log_pdf

In [ ]:
def calc_log_likelihood(data,p,m,s):
    n = data.shape[0]
    c = len(p)
    log_l =0
    for i in range(n):
        t =0
        for k in range(c):
            t += p[k]* mvn.pdf(data[i],m[k],s[k])

        log_l+=np.log(t)

    return log_l

In [ ]:
seed = 2

In [ ]:
data_x= loadmat('data/data.mat')['data']
data_y = loadmat('data/label.mat')['trueLabel'].flatten()

In [ ]:
data_xt = data_x.T

In [ ]:
x_mean = np.mean(data_xt,axis=0)
x_c = data_xt-x_mean

pca = PCA(4)
data_pca = pca.fit_transform(x_c)

m,n = data_pca.shape
c=2

pi = np.random.random(c)
pi = pi/np.sum(pi)
u_s = np.random.randn(c, 4)
mu_old = u_s.copy()

sigma = []
for ii in range(c):
    dummy = np.random.randn(4,4)
    sigma.append(dummy@dummy.T)


In [ ]:

tau = np.full((m, c), fill_value=0.)
tol = 1e-3
log_vals = []
total_itters = 0
for ii in range(100):
    total_itters+=1
    # E-step    
    for kk in range(c):
        tau[:, kk] = pi[kk] *  mvn.pdf(data_pca, u_s[kk], sigma[kk])
    # normalize tau
    sum_tau = np.sum(tau, axis=1)
    sum_tau.shape = (m,1)    
    tau = np.divide(tau, np.tile(sum_tau, (1, c)))


    for kk in range(c):
       
        pi[kk] = np.sum(tau[:, kk])/m
        
        
        u_s[kk] = data_pca.T @ tau[:,kk] / np.sum(tau[:,kk], axis = 0)
     
        dummy = data_pca - np.tile(u_s[kk], (m,1)) 
        sigma[kk] = dummy.T @ np.diag(tau[:,kk]) @ dummy / np.sum(tau[:,kk], axis = 0)

    log_vals.append(calc_log_likelihood(data_pca,pi,u_s,sigma))


    if np.linalg.norm(u_s-mu_old) < tol:
        print('training coverged')
        break
    mu_old = u_s.copy()
    
plt.xlabel("Iterations")
plt.ylabel("Log Likelihood")
plt.title(f"Log Likelihood Verse Number of Iterations\n Total Iterations: {total_itters}")
plt.plot(log_vals)

Q3.2

In [ ]:
pi

In [ ]:
u_s.shape

In [ ]:
mapping = pca.inverse_transform(u_s)

In [ ]:
reshaped_img_1 = mapping[0].reshape(28,28)
reshaped_img_2 = mapping[1].reshape(28,28)

In [ ]:
plt.imshow(reshaped_img_1, cmap='gray')
plt.title("Component 1 Mean")

In [ ]:
plt.imshow(reshaped_img_2, cmap='gray')
plt.title("Component 2 Mean")

In [ ]:
plt.imshow(sigma[0], cmap='gray')
plt.title("Component 1 Covariance")
plt.colorbar()

In [ ]:
plt.imshow(sigma[1], cmap='gray')
plt.title("Component 2 Covariance")
plt.colorbar()

Q3.3

In [ ]:
gmm_labels = np.argmax(tau,axis=1)

In [ ]:
label_mapping_dict_gmm = {}
for cluster in [0,1]:
    counts = np.unique(data_y[gmm_labels == cluster],return_counts=True)
    true_index = np.argmax(counts[1])
    label_mapping_dict_gmm[cluster]=counts[0][true_index].real


mapped_labels_gmm = np.array([label_mapping_dict_gmm[x] for x in gmm_labels])

In [ ]:
eval = data_y==mapped_labels_gmm

In [ ]:
conf_matrix_gmm = confusion_matrix(data_y,mapped_labels_gmm,labels=[2,6])

In [ ]:
miss_class_gmm = 1-np.diag(conf_matrix_gmm)/conf_matrix_gmm.sum(axis=1)

In [ ]:
kmeans = KMeans(n_clusters=2).fit(data_pca.real)

In [ ]:
k_means_labels = kmeans.labels_

label_mapping_dict_k_means = {}
for cluster in [0,1]:
    counts = np.unique(data_y[k_means_labels == cluster],return_counts=True)
    true_index = np.argmax(counts[1])
    label_mapping_dict_k_means[cluster]=counts[0][true_index].real


mapped_labels_k_means= np.array([label_mapping_dict_k_means[x] for x in k_means_labels])

eval_kmeans = data_y==mapped_labels_k_means
conf_matrix_k_means = confusion_matrix(data_y,mapped_labels_k_means,labels=[2,6])
miss_class_k_means = 1-np.diag(conf_matrix_k_means)/conf_matrix_k_means.sum(axis=1)

In [ ]:
conf_matrix_gmm,miss_class_gmm

In [ ]:
conf_matrix_k_means,miss_class_k_means